# Lab 9: Named Entity Recognition

Guidelines: https://github.com/MohammedMosuily/NLP_Lab9

Run cells from top to bottom from this folder.

In [1]:
import spacy
from spacy import displacy
from spacy.training import Example
from spacy.training.iob_utils import offsets_to_biluo_tags
from spacy.util import fix_random_seed

nlp = spacy.load("en_core_web_sm")

## Task 1: Find entities in a sentence

Set `sentence` below to any sentence you want to analyze. A default example keeps Run All non-interactive. Start and end positions below are token positions; the end position is excluded.

In [2]:
sentence = "Apple was founded by Steve Jobs in California."
print("Sentence:", sentence)
doc = nlp(sentence)

if not doc.ents:
    print("No named entities found.")

for ent in doc.ents:
    print("Entity:", ent.text)
    print("Label:", ent.label_)
    print("Start:", ent.start, "End:", ent.end)
    print("Meaning:", spacy.explain(ent.label_))
    print()

Sentence: Apple was founded by Steve Jobs in California.
Entity: Apple
Label: ORG
Start: 0 End: 1
Meaning: Companies, agencies, institutions, etc.

Entity: Steve Jobs
Label: PERSON
Start: 4 End: 6
Meaning: People, including fictional

Entity: California
Label: GPE
Start: 7 End: 8
Meaning: Countries, cities, states



## Task 2: Add missing entities and count them

The model misses `Bcd LLC` and includes extra words before `Abc & Co.`.
Add rules for the two company names to fix these results.

In [3]:
paragraph = (
    "This agreement is made and entered into by and between ‘Abc & Co.’  and ‘Bcd LLC’ "
    "for term 1 year starting from April 1, 2020,  hereinafter collectively referred "
    "to as the Parties. Samuel P. Jackson in the place (New York) and on the date "
    "written below, with the following terms and conditions."
)
print("Before:", [(ent.text, ent.label_) for ent in nlp(paragraph).ents])

ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})
ruler.add_patterns([
    {"label": "ORG", "pattern": "Abc & Co."},
    {"label": "ORG", "pattern": "Bcd LLC"},
])
doc = nlp(paragraph)
print("After:", [(ent.text, ent.label_) for ent in doc.ents])
print("PERSON count:", sum(ent.label_ == "PERSON" for ent in doc.ents))
print("ORG count:", sum(ent.label_ == "ORG" for ent in doc.ents))

assert [(ent.text, ent.label_) for ent in doc.ents if ent.label_ == "ORG"] == [
    ("Abc & Co.", "ORG"), ("Bcd LLC", "ORG")
]
displacy.render(doc, style="ent", jupyter=True)

Before: [('between ‘Abc & Co.', 'ORG'), ('1 year', 'DATE'), ('April 1, 2020', 'DATE'), ('Samuel P. Jackson', 'PERSON'), ('New York', 'GPE')]
After: [('Abc & Co.', 'ORG'), ('Bcd LLC', 'ORG'), ('1 year', 'DATE'), ('April 1, 2020', 'DATE'), ('Samuel P. Jackson', 'PERSON'), ('New York', 'GPE')]
PERSON count: 1
ORG count: 2


There is **1 PERSON** and **2 ORG** entities in the corrected paragraph.

## Task 3: Train a custom NER model

Annotate the company, people, places, and products in the given text.
Train for 20 iterations and save the model.

In [4]:
text = (
    "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. "
    "It is known for its Windows operating system and Office software products."
)
entities = [
    ("Microsoft", "ORG"),
    ("Bill Gates", "PERSON"),
    ("Paul Allen", "PERSON"),
    ("Albuquerque", "GPE"),
    ("New Mexico", "GPE"),
    ("Windows", "PRODUCT"),
    ("Office", "PRODUCT"),
]
annotations = {"entities": [
    (text.index(name), text.index(name) + len(name), label)
    for name, label in entities
]}

fix_random_seed(42)
custom_nlp = spacy.blank("en")
ner = custom_nlp.add_pipe("ner")
for name, label in entities:
    ner.add_label(label)

example = Example.from_dict(custom_nlp.make_doc(text), annotations)
assert "-" not in offsets_to_biluo_tags(example.predicted, annotations["entities"])
optimizer = custom_nlp.initialize(lambda: [example])

for iteration in range(20):
    losses = {}
    example = Example.from_dict(custom_nlp.make_doc(text), annotations)
    custom_nlp.update([example], sgd=optimizer, losses=losses)
    print(f"Iteration {iteration + 1}: {losses['ner']:.4f}")

custom_nlp.to_disk("custom_ner_model")

Iteration 1: 16.4444
Iteration 2: 26.4833
Iteration 3: 26.2558
Iteration 4: 25.8843
Iteration 5: 25.2662
Iteration 6: 24.3288
Iteration 7: 22.9318
Iteration 8: 20.6413
Iteration 9: 17.4721
Iteration 10: 13.4208
Iteration 11: 9.5739
Iteration 12: 7.6619
Iteration 13: 6.7663
Iteration 14: 5.9038
Iteration 15: 4.8513
Iteration 16: 29.7098
Iteration 17: 9.2642
Iteration 18: 8.3172
Iteration 19: 6.6880
Iteration 20: 4.9263


In [5]:
saved_nlp = spacy.load("custom_ner_model")
sentence = "Bill Gates founded Microsoft in New Mexico."
print(sentence)
for ent in saved_nlp(sentence).ents:
    print(ent.text, ent.label_)

assert [(ent.text, ent.label_) for ent in saved_nlp(text).ents] == [
    (ent.text, ent.label_) for ent in custom_nlp(text).ents
]

Bill Gates founded Microsoft in New Mexico.
Bill Gates PERSON
New Mexico GPE


This run finds Bill Gates and New Mexico but misses Microsoft. Only one sentence
was used for training, so the model does not work well on new sentences.